In [0]:
create connection if not exists earthquake_project_conn
type HTTP
options (
  host =  'https://earthquake.usgs.gov',
  port = 443,
  base_path = '/earthquakes/feed/v1.0/',
  bearer_token = 'na'
)

In [0]:
%python
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

conn = w.connections.get("earthquake_project_conn")
print(conn)
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
%python
dbutils.widgets.text('catalog_name', 'project_dev', 'project_dev')
catalog_name = dbutils.widgets.get('catalog_name')
print(catalog_name)

In [0]:
%python
spark.sql(f"use catalog {catalog_name}")
spark.sql("use schema bronze")
spark.sql("create volume if not exists earthquake_data")

In [0]:
use catalog project_dev;
use schema bronze;
create volume if not exists earthquake_data;

In [0]:
%python
import requests
import json


url = f'{base_url}summary/all_day.geojson'
response = requests.get(url)
data= response.json()
dbutils.fs.put('/Volumes/project_dev/bronze/earthquake_data/earthquake.json',json.dumps(data), overwrite=True)

In [0]:
%python
import requests
import json
import datetime

url = f"{base_url}summary/all_day.geojson"
response = requests.get(url)

if response.status_code !=200:
    raise Exception(f"Error in getting data from {url}")

data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")

dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json",
    json.dumps(data),
    overwrite=True
)
       